In [1]:
import numpy as np

# ============================================================
# Settings
# ============================================================

GRAM_FILE = "gram841.txt"
DIM = 12
TOL = 1e-7

# ============================================================
# Read Gram matrix
# ============================================================

G = np.loadtxt(GRAM_FILE)

n = G.shape[0]

print(f"N = {n}")

# Symmetrize just in case
G = 0.5 * (G + G.T)

# ============================================================
# Recover coordinates from Gram matrix
# ============================================================

eigvals, eigvecs = np.linalg.eigh(G)

idx = np.argsort(eigvals)[::-1]
eigvals = eigvals[idx]
eigvecs = eigvecs[:, idx]

print("\nTop 20 eigenvalues:")
for i in range(min(20, len(eigvals))):
    print(f"{i+1:2d}: {eigvals[i]: .16e}")

print("\n13th eigenvalue:")
print(eigvals[12])

# Top-D spectral factorization

lam = eigvals[:DIM]
U = eigvecs[:, :DIM]

X = U * np.sqrt(np.maximum(lam, 0.0))

# ============================================================
# Normalize rows
# ============================================================

row_norms_before = np.linalg.norm(X, axis=1)

X = X / row_norms_before[:, None]

row_norms_after = np.linalg.norm(X, axis=1)

max_norm_error = np.max(np.abs(row_norms_after - 1.0))

# ============================================================
# Recompute Gram matrix
# ============================================================

H = X @ X.T

# ============================================================
# Maximum off-diagonal inner product
# ============================================================

H_off = H.copy()
np.fill_diagonal(H_off, -np.inf)

i, j = np.unravel_index(np.argmax(H_off), H_off.shape)

max_offdiag = H_off[i, j]

# ============================================================
# Reconstruction error
# ============================================================

max_reconstruction_error = np.max(np.abs(H - G))

fro_error = np.linalg.norm(H - G, ord='fro')

# ============================================================
# Rank checks
# ============================================================

rank_1e8 = np.sum(eigvals > 1e-8)
rank_1e10 = np.sum(eigvals > 1e-10)
rank_1e12 = np.sum(eigvals > 1e-12)

# ============================================================
# Save coordinates
# ============================================================

np.savetxt(
    "gram841_coordinates.txt",
    X,
    fmt="%.17g"
)

# ============================================================
# Report
# ============================================================

print("\n" + "="*60)
print("RECOVERY REPORT")
print("="*60)

print(f"Recovered dimension              : {DIM}")

print(f"Rank (>1e-8)                     : {rank_1e8}")
print(f"Rank (>1e-10)                    : {rank_1e10}")
print(f"Rank (>1e-12)                    : {rank_1e12}")

print()

print(f"Maximum norm error               : {max_norm_error:.16e}")

print(f"Maximum reconstruction error     : {max_reconstruction_error:.16e}")

print(f"Frobenius reconstruction error   : {fro_error:.16e}")

print()

print(f"Maximum off-diagonal inner prod. : {max_offdiag:.16e}")

print(f"Pair (0-based)                   : ({i}, {j})")

print(f"Pair (1-based)                   : ({i+1}, {j+1})")

print(f"Gap below 1/2                    : {0.5 - max_offdiag:.16e}")

print()

if max_offdiag <= 0.5 + TOL and max_norm_error <= TOL:
    print("CONCLUSION:")
    print(
        f"The recovered configuration consists of {n} unit vectors "
        f"in R^{DIM} whose maximum off-diagonal inner product "
        f"is at most 0.5 up to tolerance {TOL:g}."
    )
    print("Numerically, this is a kissing arrangement.")
else:
    print("CONCLUSION:")
    print("Numerical kissing condition failed.")

print("\nCoordinates saved to:")
print("gram841_coordinates.txt")

N = 841

Top 20 eigenvalues:
 1:  7.5909911218787627e+01
 2:  7.5597244736097650e+01
 3:  7.5058910767139864e+01
 4:  7.4726079887934020e+01
 5:  7.3569356126235846e+01
 6:  7.3002273563543596e+01
 7:  7.2814650414113174e+01
 8:  7.2091458094693522e+01
 9:  6.3287296477066192e+01
10:  6.2174898215557498e+01
11:  6.1561557576661500e+01
12:  6.1206362922170477e+01
13:  1.6583706514054629e-11
14:  1.6189489512814708e-11
15:  1.6147692280625746e-11
16:  1.5979816492250190e-11
17:  1.5898774553193699e-11
18:  1.5819856694489279e-11
19:  1.5659259722344775e-11
20:  1.5630458836659318e-11

13th eigenvalue:
1.658370651405463e-11

RECOVERY REPORT
Recovered dimension              : 12
Rank (>1e-8)                     : 12
Rank (>1e-10)                    : 12
Rank (>1e-12)                    : 395

Maximum norm error               : 2.2204460492503131e-16
Maximum reconstruction error     : 6.5462251504916358e-13
Frobenius reconstruction error   : 2.3960489063200548e-10

Maximum off-diagonal inne